# EDA – AISHELL-5 Dataset Exploration

**Objective:** Load AISHELL-5 dev set, verify data structure, analyze statistics.

Tasks covered: T04

In [ ]:
import sys
sys.path.insert(0, "../")

import torchaudio
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.pipeline.data_loader import AISHELL5Loader

%matplotlib inline
print("Imports OK")

## 1. Load Dataset

In [ ]:
DATA_DIR = "../data/dev"
loader = AISHELL5Loader(DATA_DIR)
print(f"Total samples: {len(loader)}")
print(f"Statistics: {loader.statistics()}")

## 2. Verify 4-Channel Audio Structure

In [ ]:
# Load first sample and verify shape
sample = loader[0]
waveform, sr = torchaudio.load(str(sample.wav_path))

print(f"Session ID: {sample.session_id}")
print(f"WAV path: {sample.wav_path}")
print(f"Waveform shape: {waveform.shape}  # Expected [4, T]")
print(f"Sample rate: {sr} Hz  # Expected 16000")
print(f"Duration: {waveform.shape[-1]/sr:.2f}s")

# Assertion checks
assert waveform.shape[0] == 4, f"Expected 4 channels, got {waveform.shape[0]}"
assert sr == 16000, f"Expected 16000 Hz, got {sr}"
print("✓ Shape and SR verified")

## 3. Waveform Visualization (3 samples)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 8))
channel_labels = ["Ch0 (Driver)", "Ch1 (Passenger F)", "Ch2 (Rear L)", "Ch3 (Rear R)"]

for row_idx in range(3):
    sample = loader[row_idx]
    waveform, sr = torchaudio.load(str(sample.wav_path))
    t = np.linspace(0, waveform.shape[-1]/sr, waveform.shape[-1])
    
    for ch in range(4):
        ax = axes[row_idx][ch]
        ax.plot(t[::100], waveform[ch].numpy()[::100], linewidth=0.5, alpha=0.8)
        ax.set_title(f"Sample {row_idx+1} – {channel_labels[ch]}", fontsize=8)
        ax.set_xlabel("Time (s)", fontsize=7)
        ax.set_ylabel("Amplitude", fontsize=7)
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig("../outputs/figures/eda_waveforms.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/figures/eda_waveforms.png")

## 4. Transcript Format Analysis

In [ ]:
# Inspect transcript format
for i in range(min(3, len(loader))):
    s = loader[i]
    print(f"\n=== {s.session_id} ===")
    print(f"  n_speakers: {s.n_speakers}")
    for spk, text in s.references.items():
        print(f"  {spk}: {text[:60]}..." if len(text) > 60 else f"  {spk}: {text}")

## 5. Dataset Statistics

In [ ]:
stats_rows = []
for sample in loader:
    try:
        waveform, sr = torchaudio.load(str(sample.wav_path))
        dur = waveform.shape[-1] / sr
        stats_rows.append({
            "session_id": sample.session_id,
            "n_speakers": sample.n_speakers,
            "duration_sec": round(dur, 2),
            "n_channels": waveform.shape[0],
            "has_transcript": bool(sample.references),
        })
    except Exception as e:
        print(f"Error: {sample.session_id}: {e}")

df = pd.DataFrame(stats_rows)
print(f"Total recordings: {len(df)}")
print(f"\nSpeaker distribution:")
print(df["n_speakers"].value_counts().sort_index())
print(f"\nDuration stats:")
print(df["duration_sec"].describe())
print(f"\nTranscripts available: {df['has_transcript'].sum()} / {len(df)}")

# Save statistics
df.to_csv("../outputs/metrics/eda_stats.csv", index=False)
print("\nSaved: outputs/metrics/eda_stats.csv")

In [ ]:
# Duration distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["duration_sec"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribution of Recording Duration")
axes[0].set_xlabel("Duration (s)")
axes[0].set_ylabel("Count")

axes[1].bar(
    df["n_speakers"].value_counts().sort_index().index,
    df["n_speakers"].value_counts().sort_index().values,
    color="coral", edgecolor="white"
)
axes[1].set_title("Distribution of Number of Speakers")
axes[1].set_xlabel("Number of Speakers")
axes[1].set_ylabel("Count")
axes[1].set_xticks([2, 3, 4])

plt.tight_layout()
plt.savefig("../outputs/figures/eda_statistics.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/figures/eda_statistics.png")

## Summary

- ✅ 4-channel WAV files confirmed: shape `[4, T]`, 16kHz
- ✅ Transcript format: SPK1/SPK2... with Chinese text
- ✅ Duration distribution analyzed
- ✅ Statistics saved to CSV